# Train a TransferEncoder

Trains the encoder of `transfer_models.TransferEncoder` (see `DESIGN-Transfer.md`): a frozen backbone (`vgg19`, `vgg19_keypoints` or `vit_b_16`), a trainable projection to the latent, and one regression head for each robot of `exp["robots"]`. With one robot, this is the single-source encoder of the paper. With several robots, it is design D1, and with the `vgg19_keypoints` backbone it is design D2. With `robot_mask`, the background is removed (D3): replaced by gray, or swapped for random backgrounds during training.

Every training step takes one batch from each robot; the loss is the sum of the losses of the robots. The model with the best validation loss (on the `validation_data` of the robots) is kept.

In [ ]:
import sys
sys.path.append("../..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import math
import pathlib
import torch
import torch.nn as nn

import training_harness as helper_training
from exprunflow.reproducibility import configure_deterministic_run
from visual_proprioception.transfer.transfer_data import (
    backgrounds_of_items, foreground_masks, load_frames, masked_batch, synthetic_backgrounds)
from visual_proprioception.transfer.transfer_models import TransferEncoder

In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# If it is set to discard-old, the exprun will be recreated from scratch
creation_style = "exist-ok"

experiment = "sensorprocessing_transfer_encoder"
run = "vgg19_128_panda"
# If not None, set the epochs to something different than the exp
epochs = None

# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

In [ ]:
if expruns_path:
    Config().set_exprun_path(pathlib.Path(expruns_path))
if results_path:
    Config().set_results_path(pathlib.Path(results_path))

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
seed = configure_deterministic_run(exp["seed"])
device = Config().runtime["device"]

In [ ]:
# The frames of every robot. With a robot mask (D3), also the masks against the
# median background of each demonstration.
will_load_existing = helper_training.model_available(exp) and exp.get("reload_existing_model", True)
data = {}
if not will_load_existing:
    for robot in exp["robots"]:
        exp_robot = Config().get_experiment(robot["robot_exp"], robot["robot_run"])
        for split in ["training_data", "validation_data"]:
            frames = load_frames(robot[split], exp["image_size"], exp_robot, exp["frame_step"])
            frames["masks"] = None
            if exp.get("robot_mask"):
                frames["masks"] = foreground_masks(
                    frames["images"], frames["demo_index"],
                    exp["robot_mask"]["threshold"], exp["robot_mask"]["dilate"])
            data[(robot["name"], split)] = frames
            print(f'{robot["name"]} {split}: {frames["images"].shape[0]} frames')

# the backgrounds of the background swap: synthetic ones, and the median
# backgrounds of demonstrations of robots that are neither sources nor the target
generator = torch.Generator().manual_seed(seed)
backgrounds = None
if not will_load_existing and exp.get("robot_mask") and exp["robot_mask"]["fill"] == "background_swap":
    backgrounds = synthetic_backgrounds(exp["synthetic_backgrounds"], exp["image_size"], generator)
    if exp.get("background_swap_items"):
        backgrounds = torch.cat([backgrounds, backgrounds_of_items(exp["background_swap_items"], exp["image_size"])])
    print(f"{backgrounds.shape[0]} backgrounds for the swap")

In [ ]:
batch_size = exp["batch_size_per_robot"]
names = [robot["name"] for robot in exp["robots"]]


def make_batch(frames, rows, training):
    """A batch on the device. The swap and the color jitter are applied only in training."""
    masks = None if frames["masks"] is None else frames["masks"][rows]
    x = masked_batch(frames["images"][rows], masks,
                     backgrounds=backgrounds if training else None,
                     jitter=exp["color_jitter"] if training else 0.0,
                     generator=generator)
    return x.to(device), frames["targets"][rows].to(device)


def model_training_step(model, optimizer):
    """One epoch: every step takes a batch of every robot; the loss is the sum of their losses"""
    model.train()
    sizes = {name: data[(name, "training_data")]["images"].shape[0] for name in names}
    steps = max(math.ceil(size / batch_size) for size in sizes.values())
    total = 0
    for _ in range(steps):
        optimizer.zero_grad()
        loss = 0
        for name in names:
            rows = torch.randint(sizes[name], (min(batch_size, sizes[name]),), generator=generator)
            x, y = make_batch(data[(name, "training_data")], rows, True)
            loss = loss + criterion(model(x, name), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / steps


def model_eval_step(model):
    """The validation loss, averaged over the robots"""
    model.eval()
    losses = []
    with torch.no_grad():
        for name in names:
            frames = data[(name, "validation_data")]
            n = frames["images"].shape[0]
            total = 0
            for i in range(0, n, batch_size):
                rows = torch.arange(i, min(i + batch_size, n))
                x, y = make_batch(frames, rows, False)
                total += criterion(model(x, name), y).item() * len(rows)
            losses.append(total / n)
    return sum(losses) / len(losses)

In [ ]:
model = TransferEncoder(exp)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=exp["learning_rate"])
model = helper_training.load_or_train(
    exp, model, optimizer, model_training_step, model_eval_step, epochs=epochs)

In [ ]:
exp.done()